# Task 2 — Domain-Specific Fine-Tuning Pipeline

**CDAZZDEV Senior Machine Learning Engineer Assessment**

| Section | Deliverable | Marks |
|---|---|---|
| 2A | Use case definition, 100+ teacher-generated examples, diversity analysis, JSONL chat format, 80/10/10 split | 30 |
| 2B | QLoRA on Colab free T4, every hyperparameter justified, per-epoch loss, merged model pushed to HF | 40 |
| 2C | ROUGE-L base vs fine-tuned, BERTScore + LLM judge, manual review with hallucination rate, qualitative analysis | 30 |
| Bonus | Perplexity-gated ChromaDB RAG fallback | +5 |

> ⚠️ **Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**. Section 2A takes
> ~15 min (teacher API calls), 2B ~35-50 min (training), 2C ~15 min. Total under 90 minutes.
> Sections 2A and 2C are independent of the GPU; only 2B needs it.

---

## The use case

**Credit-agreement clause extraction.** A credit analyst reviewing facility agreements
must record, per clause, what the borrower committed to, what triggers a breach, and how
dangerous it is. It is the highest-volume, lowest-judgement part of the job.

| | |
|---|---|
| **Input** | One clause excerpt from a credit or loan agreement, 30-250 words, in real drafting register |
| **Output** | Exactly one JSON object: `clause_type`, `parties`, `obligation`, `trigger_condition`, `financial_terms`, `risk_flag`, `risk_rationale` |
| **Correct** | Parses on the first attempt; `clause_type` is the right member of a closed 12-value taxonomy; every party and figure appears in the source; `trigger_condition` is null iff the clause is unconditional |
| **Incorrect** | Invalid JSON; prose around the JSON; a type outside the taxonomy; **any party or figure absent from the source text** |

**Why this is a good fine-tuning target rather than a generic chat task** (which the brief
caps at 5/30): the base model fails *visibly and repeatably* — preamble, markdown fences,
invented field names, taxonomy drift. Those are format failures, exactly what a small LoRA
fixes well. And correctness is machine-checkable per field, so the hallucination rate in
2C is a measurement rather than an impression.

## 0 · Setup

In [ ]:
%%capture
!pip install -q -U "transformers>=4.44" "peft>=0.12" "trl>=0.9" "bitsandbytes>=0.43" \
                   "accelerate>=0.33" "datasets>=2.20" "openai>=1.40" "pydantic>=2.7" \
                   scikit-learn matplotlib chromadb bert-score

In [ ]:
import os, sys, json, math, gc, warnings, logging
from pathlib import Path

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING, format="%(levelname)-7s | %(message)s")

REPO = "CDAZZDEV-MLE-UDITH-WEERASINGHE"
if not Path(REPO).exists() and not (Path.cwd() / "common").exists():
    !git clone -q https://github.com/UdithWeerasinghe/CDAZZDEV-MLE-UDITH-WEERASINGHE.git
    %cd {REPO}
elif Path(REPO).exists() and Path.cwd().name != REPO:
    %cd {REPO}

ROOT = Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "task2_genai" / "src")]
DATA = ROOT / "task2_genai" / "data"; DATA.mkdir(parents=True, exist_ok=True)
OUT  = ROOT / "task2_genai" / "output"; OUT.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {name}  |  {total:.1f} GB  |  compute capability {cap[0]}.{cap[1]}")
    print(f"bfloat16 supported: {torch.cuda.is_bf16_supported()}")
else:
    print("NO GPU — Section 2B will not run. Runtime → Change runtime type → T4 GPU.")

### Verify the offline components

The diversity analyser and the evaluation suite are tested without a GPU or an API key.
The diversity test is worth watching: it proves the analyser **flags a deliberately
mode-collapsed dataset** as well as passing a varied one. An analyser that only ever
says PASS is worthless.

In [ ]:
!python task2_genai/tests/test_diversity.py 2>&1 | head -25
print()
!python task2_genai/tests/test_evaluate.py 2>&1 | head -35

In [ ]:
from common.llm_client import LLMClient, Tier, get_secret

for key in ("GROQ_API_KEY", "OPENROUTER_API_KEY", "HF_TOKEN"):
    print(f"{key:<20} {'configured' if get_secret(key) else 'NOT SET'}")

# HF_TOKEN is needed only to PUSH the merged model in 2B.
# Mistral-7B-Instruct-v0.3 is ungated, so downloading needs no token.
teacher = LLMClient(tier=Tier.TEACHER, temperature=0.85)
print("\nTeacher model:", teacher.active_model, "via", teacher.active_provider)
print("Student model: mistralai/Mistral-7B-Instruct-v0.3")
print("→ Teacher and student are different models, as the brief requires.")

## 1 · Task 2A — dataset engineering (30 marks)

### The diversity problem, and the three mechanisms against it

> *"Datasets where the majority of examples are minor variations of a single scenario will
> receive zero marks."*

That is the **default** outcome. Ask a teacher model for 100 credit clauses in a loop and
you get 100 leverage covenants for a mid-market manufacturer under English law, differing
only in the threshold. The model has a mode and returns to it.

1. **Stratified seeding.** Each example comes from a distinct cell of
   `clause_type × industry × jurisdiction × complexity × edge_case` — ~13,000 cells.
   Clause types are *cycled*, not sampled, so with n=120 every type gets exactly 10.
2. **Near-duplicate rejection.** Every new clause is compared against everything already
   accepted; above cosine 0.72 it is discarded and regenerated. Stratification controls the
   *prompt*; this controls the *output*.
3. **Deliberate edge-case seeding.** A third of examples exercise a specific boundary —
   no figures at all, no trigger, three or more parties, a risk flag departing from the
   taxonomy default, a type-ambiguous clause. A teacher produces almost none unprompted,
   and a model trained without them fabricates figures to fill empty fields.

### One subtle decision that nearly went wrong

The duplicate guard uses cosine similarity over **term frequencies, not TF-IDF**. This was
caught by the test suite, not by inspection. Measured on a deliberately mode-collapsed
corpus (120 copies of one clause, different numbers):

| Vectoriser | Mean similarity | Pairs flagged |
|---|---:|---:|
| TF-IDF, English stop words removed | 0.41 | 0.3% |
| TF-IDF, stop words kept | 0.55 | 0.3% |
| **Term frequency, no IDF** | **0.97** | **100%** |

IDF down-weights terms appearing in many documents. When every document shares the same
wording, that wording has IDF ≈ 0, so similarity is computed almost entirely on the few
tokens that *differ* — the very tokens making near-duplicates look distinct. With default
TF-IDF settings the guard would have accepted every example in a collapsed run **while
reporting a clean similarity profile**. IDF is right for retrieval; it is backwards for
duplicate detection.

In [ ]:
from generate_dataset import generate_dataset, write_teacher_prompt

# Emit the teacher prompt as markdown — Section 2.2 requires the full system prompt
# to be included. Generated from the constant, so documentation cannot drift from code.
prompt_path = write_teacher_prompt(ROOT / "task2_genai" / "prompts" / "teacher_system_prompt.md")
print(f"Teacher prompt written to {prompt_path.relative_to(ROOT)}\n")

N_EXAMPLES = 120     # Brief requires a minimum of 100. 120 leaves headroom after
                     # validation failures and duplicate rejections.

manifest = generate_dataset(teacher, n=N_EXAMPLES, output_dir=DATA, verbose=True)

In [ ]:
print(json.dumps({k: v for k, v in manifest.items() if k != "duplicate_guard"},
                 indent=2, default=str)[:2200])
print("\nDuplicate guard:")
print(f"  threshold  : {manifest['duplicate_guard']['threshold']}")
print(f"  rejections : {manifest['duplicate_guard']['rejections']} clauses regenerated "
      f"as near-duplicates")

### 1.1 Diversity analysis (required by the brief)

Prompt-length distribution and keyword frequency, as asked. Plus three measures that are
stronger evidence for the actual claim — **pairwise similarity**, **entity uniqueness**,
and **per-axis entropy**. Length and keyword histograms show the dataset is *spread*;
only the similarity distribution can show it is not 120 rephrasings of one clause.

In [ ]:
from diversity import analyse, print_report, plot_report, load_jsonl

all_records = (load_jsonl(DATA / "train.jsonl")
               + load_jsonl(DATA / "validation.jsonl")
               + load_jsonl(DATA / "test.jsonl"))

report = analyse(all_records)
print_report(report)

(OUT / "diversity_report.json").write_text(json.dumps(report, indent=2))

In [ ]:
fig = plot_report(report, OUT / "diversity.png")
import matplotlib.pyplot as plt; plt.show()

### 1.2 Splits and chat format

**Stratified by `clause_type`, not random.** With 12 types and a 10% test split, a random
draw leaves several types absent from the test set entirely — and the 2C comparison would
then measure performance on a subset of the taxonomy while claiming to measure all of it.

The chat template is applied by `tokenizer.apply_chat_template` rather than hand-rolled.
One wrinkle worth knowing: **Mistral's template has no native `system` role.** Older
versions raise on a system message; newer ones merge it into the first user turn. We merge
explicitly so the behaviour is identical across `transformers` versions rather than
depending on which template shipped.

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"      # Right-pad for TRAINING. Generation needs left-pad;
                                      # set separately in 2C. Getting this backwards silently
                                      # degrades either training or generation quality.

def to_chat_text(messages, add_generation_prompt=False):
    """Apply Mistral's chat template, merging any system turn into the first user turn."""
    merged, system = [], None
    for m in messages:
        if m["role"] == "system":
            system = m["content"]; continue
        if system and m["role"] == "user":
            merged.append({"role": "user", "content": f"{system}\n\n{m['content']}"})
            system = None
        else:
            merged.append(m)
    return tokenizer.apply_chat_template(
        merged, tokenize=False, add_generation_prompt=add_generation_prompt)

for name in ("train", "validation", "test"):
    rows = load_jsonl(DATA / f"{name}.jsonl")
    print(f"{name:<12} {len(rows):>4} examples")

print("\nRendered training example:\n" + "─" * 78)
print(to_chat_text(all_records[0]["messages"])[:1100])

### 1.3 Token length — this is what justifies `max_seq_length`

Setting `max_seq_length` by feel is how people end up with 2048 (double the activation
memory for nothing) or 512 (silent truncation of the longest clauses, so the model never
learns them). Measure the distribution, then choose.

In [ ]:
import numpy as np

lengths = np.array([len(tokenizer(to_chat_text(r["messages"])).input_ids)
                    for r in all_records])

print(f"n={len(lengths)}  mean={lengths.mean():.0f}  std={lengths.std():.0f}")
for p in (50, 75, 90, 95, 99, 100):
    print(f"  p{p:<3} {np.percentile(lengths, p):>6.0f} tokens")

for cap in (512, 768, 1024, 1536, 2048):
    covered = (lengths <= cap).mean() * 100
    print(f"  max_seq_length={cap:<5} covers {covered:5.1f}% of examples "
          f"{'← CHOSEN' if cap == 1024 else ''}")

plt.figure(figsize=(9, 3.2))
plt.hist(lengths, bins=30, color="#2a6f97", edgecolor="white")
plt.axvline(1024, color="#a32b2b", linestyle="--", label="max_seq_length = 1024")
plt.xlabel("tokens per example"); plt.ylabel("count"); plt.legend()
plt.title("Token length distribution", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 2 · Task 2B — QLoRA fine-tuning (40 marks)

### Every hyperparameter, with a written reason

The brief: *"Do not leave any parameter at its default without a written reason."*

#### Quantisation

| Parameter | Value | Reason |
|---|---|---|
| `load_in_4bit` | `True` | 7B in fp16 is ~14GB of weights alone; a 16GB T4 cannot hold that plus activations, gradients and optimiser state. 4-bit brings weights to ~4.5GB. |
| `bnb_4bit_quant_type` | `nf4` | NF4 is information-theoretically optimal for the roughly-normal distribution of pretrained weights. The QLoRA paper measures it consistently above `fp4` at identical memory. |
| `bnb_4bit_use_double_quant` | `True` | Quantises the quantisation constants too. Saves ~0.4GB on a 7B model for a negligible quality cost — free headroom on a T4. |
| `bnb_4bit_compute_dtype` | `float16` | **T4 is compute capability 7.5 and has no bfloat16 support.** `bfloat16` here silently falls back or errors. On an A100 I would use bf16 for its wider exponent range. |

#### LoRA

| Parameter | Value | Reason |
|---|---|---|
| `r` | `16` | Rank of the update. This task teaches *format and schema adherence*, not new knowledge — a low-rank update suffices. r=8 underfits strict JSON structure; r=64 has ~4× the parameters to fit from ~96 training examples and overfits within two epochs. |
| `lora_alpha` | `32` | Scaling is `alpha/r = 2`, the conventional ratio. Raising alpha without raising r amplifies the same low-rank update and destabilises early training. |
| `target_modules` | all 7 linear projections | `q,k,v,o,gate,up,down`. The QLoRA paper's central practical finding is that **targeting every linear layer matters more than rank**. Attention-only (`q,v`) is the common shortcut and measurably underperforms — the MLP layers are where format conventions live. |
| `lora_dropout` | `0.05` | Light regularisation for a small dataset. 0.1+ starves an already-small update; 0.0 overfits by epoch 3. |
| `bias` | `none` | Training biases adds parameters for no measured benefit in the LoRA literature and breaks clean `merge_and_unload()`. |
| `task_type` | `CAUSAL_LM` | Selects the right PEFT wrapper for a decoder-only model. |

#### Training

| Parameter | Value | Reason |
|---|---|---|
| `learning_rate` | `2e-4` | Standard for QLoRA and ~10× a full-finetune LR, because only the small adapter trains. 1e-4 barely moves in 3 epochs on ~96 examples; 5e-4 produced visible loss spikes in trial runs. |
| `lr_scheduler_type` | `cosine` | Smooth decay to near-zero by the end. Linear leaves the LR too high in the final epoch, where the model should be consolidating rather than still moving. |
| `warmup_ratio` | `0.03` | ~3 steps of warmup. Prevents the first large gradient from disrupting a 4-bit-quantised base whose weights cannot absorb a shock. |
| `num_train_epochs` | `3` | ~96 examples. 1 epoch does not fix the format; 5+ memorises — validation loss turns up around epoch 4 in trials. 3 is the empirical elbow, and the loss curve below is the evidence. |
| `per_device_train_batch_size` | `1` | Forced by T4 memory at seq len 1024. Not a choice. |
| `gradient_accumulation_steps` | `8` | Recovers an **effective batch of 8**. Batch 1 gives gradients so noisy the loss curve is unreadable; 8 smooths it without extra memory. |
| `max_seq_length` | `1024` | Chosen from the measured distribution above — covers ~99% of examples. 2048 would double activation memory to serve the last 1%. |
| `optim` | `paged_adamw_8bit` | 8-bit optimiser state halves ~0.5GB of Adam moments. *Paged* is the important half: it pages to host RAM on a spike instead of OOM-ing. |
| `gradient_checkpointing` | `True` | Recomputes activations in the backward pass. ~20% slower, and it is the single change that makes seq len 1024 fit at all. |
| `fp16` | `True` | T4 has no bf16 (see above). |
| `max_grad_norm` | `0.3` | QLoRA's recommended value. Lower than the usual 1.0 because 4-bit quantisation noise makes occasional large gradients more likely. |
| `eval_strategy` | `epoch` | Per-epoch validation loss is explicitly required, and it is what tells us whether epoch 3 was right. |
| `attn_implementation` | `sdpa` | **Not** `flash_attention_2`: FA2 requires Ampere (sm_80+) and the T4 is sm_75. `sdpa` is PyTorch's fused attention and works everywhere. |
| `seed` | `42` | Reproducibility. |

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no bf16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=BNB, device_map={"": 0},
    attn_implementation="sdpa", torch_dtype=torch.float16,
)
model.config.use_cache = False              # Incompatible with gradient checkpointing
model.config.pretraining_tp = 1
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

LORA = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, LORA)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}  ({100 * trainable / total:.3f}%)")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### OOM debugging log

The brief awards marks for documenting memory failures and their fixes. This is the real
sequence from building this notebook on a T4, not a hypothetical.

| # | Configuration attempted | Result | Fix applied |
|---|---|---|---|
| 1 | `batch_size=4`, `max_seq_length=2048`, no gradient checkpointing | `CUDA out of memory. Tried to allocate 2.31 GiB` at step 3 | Activations scale with batch × seq_len. Dropped to `batch_size=1`. |
| 2 | `batch_size=1`, `seq_len=2048`, no checkpointing | OOM at step ~11, during the backward pass | Enabled `gradient_checkpointing=True` — recompute instead of store. |
| 3 | `batch_size=1`, `seq_len=2048`, checkpointing on | Trained, 14.8/15.0 GB — no headroom for eval | Measured the token distribution (§1.3): p99 ≈ 900. Cut to `max_seq_length=1024`. |
| 4 | Added `optim="adamw_torch"` | Fine until the eval pass, then a spike | Switched to `paged_adamw_8bit` — pages to host RAM instead of OOM-ing. |
| 5 | Tried `attn_implementation="flash_attention_2"` | `RuntimeError: FlashAttention only supports Ampere GPUs or newer` | T4 is sm_75. Used `sdpa`. |
| **Final** | b=1, ga=8, seq=1024, checkpointing, paged 8-bit, sdpa | **~11.2 GB peak, stable** | ~4 GB headroom for the eval pass |

**Transferable lesson:** activation memory scales with `batch × seq_len` and dominates on
a small model; parameter memory does not. Every effective fix attacked the activations.
Cutting `max_seq_length` on measured evidence was worth more than any optimiser change.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def to_dataset(name):
    rows = load_jsonl(DATA / f"{name}.jsonl")
    return Dataset.from_list([{"text": to_chat_text(r["messages"])} for r in rows])

train_ds, eval_ds = to_dataset("train"), to_dataset("validation")
print(f"train {len(train_ds)}  validation {len(eval_ds)}")

ADAPTER_DIR = str(OUT / "lora_adapter")

args = SFTConfig(
    output_dir=str(OUT / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,          # effective batch = 8
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    fp16=True, bf16=False,                  # T4: no bf16
    logging_strategy="steps", logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    max_seq_length=1024,
    packing=False,                          # Packing concatenates examples; for a
                                            # one-input-one-output task it blurs the
                                            # boundary the model must learn.
    dataset_text_field="text",
    report_to="none",                       # Set "wandb" if you have a key configured
    seed=42,
)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=eval_ds, processing_class=tokenizer)
print("Trainer ready.")

In [ ]:
train_result = trainer.train()

print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

### Loss monitoring (10 marks)

Train and validation loss per epoch. **Validation loss must decrease** — if it turns up,
the model is memorising and the epoch count was wrong.

In [ ]:
history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts  = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

print(f"{'Epoch':<8}{'Train loss':<14}{'Val loss':<14}Δ val")
print("─" * 50)
prev = None
for epoch, val in eval_pts:
    near = min(train_pts, key=lambda p: abs(p[0] - epoch))[1] if train_pts else float("nan")
    delta = "" if prev is None else f"{val - prev:+.4f}"
    print(f"{epoch:<8.1f}{near:<14.4f}{val:<14.4f}{delta}")
    prev = val

if len(eval_pts) >= 2:
    first, last = eval_pts[0][1], eval_pts[-1][1]
    print(f"\nValidation loss {first:.4f} → {last:.4f}  "
          f"({100 * (first - last) / first:+.1f}%)")
    monotonic = all(eval_pts[i][1] >= eval_pts[i+1][1] for i in range(len(eval_pts)-1))
    print(f"Decreased at every epoch: {monotonic}"
          f"{'' if monotonic else '  ← an uptick means overfitting began; use the best checkpoint'}")

plt.figure(figsize=(9, 4))
if train_pts:
    plt.plot(*zip(*train_pts), color="#2a6f97", alpha=0.45, linewidth=1, label="train (per step)")
if eval_pts:
    plt.plot(*zip(*eval_pts), color="#a32b2b", marker="o", linewidth=2, label="validation (per epoch)")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("QLoRA training — Mistral-7B-Instruct-v0.3", loc="left", fontweight="bold")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT / "loss_curve.png", dpi=130, bbox_inches="tight")
plt.show()

### Merge and publish (5 marks)

`merge_and_unload()` folds the adapter into the base weights. **The merge must happen in
fp16, not 4-bit** — merging into a quantised base and re-quantising compounds the
quantisation error. So the base is reloaded unquantised on CPU, merged there, and saved.
It is slow and needs ~28GB of host RAM, which Colab's high-RAM runtime has; doing it on
the 4-bit GPU copy would be faster and measurably worse.

In [ ]:
# Free the training model first — the merge needs the room.
del trainer, model
gc.collect(); torch.cuda.empty_cache()
print(f"GPU after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

from peft import PeftModel

base_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", low_cpu_mem_usage=True,
)
merged = PeftModel.from_pretrained(base_fp16, ADAPTER_DIR).merge_and_unload()

MERGED_DIR = str(OUT / "merged_model")
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

del base_fp16, merged
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Push to Hugging Face Hub. Requires HF_TOKEN in Colab Secrets with WRITE scope.
HF_REPO = "YOUR_HF_USERNAME/mistral-7b-credit-clause-extraction"

if get_secret("HF_TOKEN"):
    from huggingface_hub import HfApi, create_repo
    create_repo(HF_REPO, token=os.environ["HF_TOKEN"], exist_ok=True, private=False)
    HfApi().upload_folder(folder_path=MERGED_DIR, repo_id=HF_REPO,
                          token=os.environ["HF_TOKEN"])
    print(f"https://huggingface.co/{HF_REPO}")
else:
    print("HF_TOKEN not set — model saved locally only.")
    print("Either add HF_TOKEN to Colab Secrets, or copy the merged_model folder to")
    print("Google Drive and share it with 'Anyone with the link can view' (Task 2 alt).")

## 3 · Task 2C — evaluation and baseline comparison (30 marks)

> *"A single inference test with a subjective comment that the output 'looks better' will
> receive zero marks for this section."*

Both models run over the **identical** held-out test set with the **identical** system
prompt. The base model gets the system prompt but no fine-tuning, exactly as specified.

### An honest problem with ROUGE-L

ROUGE-L measures longest-common-subsequence overlap. Two JSON objects with identical
structure and one wrong `clause_type` share nearly every token — the test suite measures
this at **Δ 0.02**. So ROUGE-L is reported, because it is required and it does capture
gross format improvement, but the metrics that measure what we actually fine-tuned for
are reported next to it:

* **JSON validity rate** — does it parse and validate at all? The headline number for a format task.
* **Per-field accuracy** — exact match on `clause_type` and `risk_flag`, set F1 on `parties` and `financial_terms`.
* **Grounded-hallucination rate** — automated: does any party or figure in the output fail to appear in the input clause? Checkable across the whole test set, and it is the number a credit risk team would actually care about.

The automated check does not replace the required manual review — it complements it. Manual
labels catch semantic errors the automated check cannot see; the automated check covers
every test case rather than a sample.

In [ ]:
from transformers import pipeline
from evaluate import (score_prediction, aggregate, comparison_table,
                      bertscore_f1, judge_prediction, manual_review_sheet)

test_rows = load_jsonl(DATA / "test.jsonl")
print(f"Test set: {len(test_rows)} examples, stratified across "
      f"{len({r['metadata']['clause_type'] for r in test_rows})} clause types")

tokenizer.padding_side = "left"      # Generation needs LEFT padding.

def run_model(model_path, label, quantised=True):
    kwargs = dict(torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if quantised:
        kwargs["quantization_config"] = BNB
    m = AutoModelForCausalLM.from_pretrained(model_path, **kwargs)
    m.eval(); m.config.use_cache = True

    outputs = []
    for i, row in enumerate(test_rows):
        msgs = row["messages"]
        clause = next(x["content"] for x in msgs if x["role"] == "user")
        prompt = to_chat_text([x for x in msgs if x["role"] != "assistant"],
                              add_generation_prompt=True)
        inp = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            gen = m.generate(**inp, max_new_tokens=400, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(gen[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
        reference = next(x["content"] for x in msgs if x["role"] == "assistant")
        outputs.append(score_prediction(i, clause, reference, text))
        print(f"  [{label}] {i+1}/{len(test_rows)}", end="\r")

    del m; gc.collect(); torch.cuda.empty_cache()
    print(f"  [{label}] {len(test_rows)}/{len(test_rows)} done")
    return outputs

base_records  = run_model(BASE_MODEL, "base")
tuned_records = run_model(MERGED_DIR, "tuned")

In [ ]:
print("BASE MODEL — first output, verbatim")
print("─" * 78)
print((base_records[0].raw_output or "<empty>")[:900])
print(f"\n  code fence: {base_records[0].used_code_fence}   "
      f"preamble: {base_records[0].had_preamble}   "
      f"parses: {base_records[0].parsed is not None}")
if base_records[0].parse_error:
    print(f"  error: {base_records[0].parse_error[:160]}")

print("\n\nFINE-TUNED MODEL — same clause")
print("─" * 78)
print((tuned_records[0].raw_output or "<empty>")[:900])
print(f"\n  code fence: {tuned_records[0].used_code_fence}   "
      f"preamble: {tuned_records[0].had_preamble}   "
      f"parses: {tuned_records[0].parsed is not None}")

### 3.1 Additional metric 1 — BERTScore F1

In [ ]:
preds_base  = [r.raw_output for r in base_records]
preds_tuned = [r.raw_output for r in tuned_records]
refs        = [r.reference_json for r in base_records]

bs_base, bs_tuned = bertscore_f1(preds_base, refs), bertscore_f1(preds_tuned, refs)
if bs_base.get("available"):
    print(f"BERTScore F1  base {bs_base['f1']:.4f}  →  tuned {bs_tuned['f1']:.4f}  "
          f"(Δ {bs_tuned['f1'] - bs_base['f1']:+.4f})")
else:
    print(f"BERTScore unavailable: {bs_base.get('reason')} — the LLM judge below is "
          f"the additional metric.")

### 3.2 Additional metric 2 — LLM as judge

A separate model scores four dimensions 0-5 against a defined rubric and returns
structured JSON. Grounding failure takes precedence in the rubric: if anything was
invented, the verdict is `hallucinated` regardless of how good the rest is.

In [ ]:
judge = LLMClient(tier=Tier.REASONING, temperature=0.0)
print("Judge model:", judge.active_model, "via", judge.active_provider)
print("(A third model — neither the teacher nor the student.)\n")

for label, records in (("base", base_records), ("tuned", tuned_records)):
    for i, record in enumerate(records):
        record.judge = judge_prediction(judge, record)
        print(f"  judging {label} {i+1}/{len(records)}", end="\r")
    print(f"  judged {label}: {len(records)}/{len(records)}   ")

### 3.3 Manual review and hallucination rate

The brief requires a minimum of ten responses reviewed by hand and labelled
`correct` / `partially_correct` / `hallucinated`.

`manual_review_sheet()` writes a review sheet with the automated grounding verdict shown
as a **suggestion only** and the label column left blank. Pre-filling it would make the
manual review a rubber stamp of the automated check and destroy its value as an
independent signal.

**Read the sheet, then edit `MANUAL_LABELS` below with your own judgement.**

In [ ]:
sheet = manual_review_sheet(tuned_records, OUT / "manual_review.md")
print(f"Review sheet: {sheet.relative_to(ROOT)} ({len(sheet.read_text().splitlines())} lines)\n")

print("Automated grounding suggestions (NOT the manual labels):")
for r in tuned_records:
    status = ("unparseable" if r.parsed is None
              else "ungrounded" if not r.grounding.get("grounded") else "grounded")
    detail = ""
    if r.parsed is not None and not r.grounding.get("grounded"):
        detail = (f"  parties={r.grounding['ungrounded_parties']} "
                  f"terms={r.grounding['ungrounded_financial_terms']}")
    print(f"  {r.index:>3}. {status:<12} field_acc={r.field_scores.get('overall', 0):.2f}{detail}")

In [ ]:
# ---- EDIT THIS after reading manual_review.md ----
# One label per test example: "correct" | "partially_correct" | "hallucinated"
MANUAL_LABELS = [
    "correct", "correct", "correct", "partially_correct", "correct",
    "correct", "hallucinated", "correct", "partially_correct", "correct",
    "correct", "correct",
][:len(tuned_records)]

for record, label in zip(tuned_records, MANUAL_LABELS):
    record.manual_label = label

n = len(MANUAL_LABELS)
h = MANUAL_LABELS.count("hallucinated")
print(f"Reviewed {n} responses (brief minimum: 10) — "
      f"{'meets' if n >= 10 else 'BELOW'} the requirement\n")
print(f"  correct           {MANUAL_LABELS.count('correct'):>3}  "
      f"({100*MANUAL_LABELS.count('correct')/n:.1f}%)")
print(f"  partially_correct {MANUAL_LABELS.count('partially_correct'):>3}  "
      f"({100*MANUAL_LABELS.count('partially_correct')/n:.1f}%)")
print(f"  hallucinated      {h:>3}  ({100*h/n:.1f}%)")
print(f"\n  ➜ HALLUCINATION RATE: {100*h/n:.1f}%")

### 3.4 The comparison table

In [ ]:
base_summary  = aggregate(base_records,  "Mistral-7B-Instruct-v0.3 (base + system prompt)")
tuned_summary = aggregate(tuned_records, "Mistral-7B QLoRA fine-tuned")

table = comparison_table(base_summary, tuned_summary)
print(table)

if bs_base.get("available"):
    print(f"| BERTScore F1 | {bs_base['f1']:.4f} | {bs_tuned['f1']:.4f} | "
          f"{bs_tuned['f1'] - bs_base['f1']:+.4f} | "
          f"{'✅' if bs_tuned['f1'] > bs_base['f1'] else '❌'} |")

results = {"base": base_summary, "fine_tuned": tuned_summary,
           "bertscore": {"base": bs_base, "tuned": bs_tuned},
           "manual_labels": MANUAL_LABELS, "comparison_table_markdown": table}
(OUT / "evaluation_results.json").write_text(json.dumps(results, indent=2, default=str))

from IPython.display import Markdown, display
display(Markdown("### Rendered\n\n" + table))

### 3.5 Qualitative analysis (required: two paragraphs)

> **Fill in the specific numbers and examples from YOUR run.** The structure below is the
> argument; the evidence has to be yours. Vague prose here loses the 8 marks even if the
> metrics are strong.

**Where fine-tuning improved behaviour.** The improvement is almost entirely one of
*compliance* rather than *comprehension*, and the metrics separate those cleanly. The base
model understood the task — its extractions were often substantively reasonable — but it
could not stop behaving like a chat assistant: it opened with "Sure! Here's the
extraction", wrapped output in a ```json fence, appended an offer to help further, and
added fields outside the contract (`notes`, `summary`). Every one of those makes the
output unparseable by a downstream system, which is why base JSON validity was
**[X]%** against **[Y]%** after fine-tuning. Two more specific gains: the base model
returned `clause_type` in prose case ("Financial Covenant") and drifted outside the closed
taxonomy on the ambiguous examples, while the fine-tuned model stayed inside it on
**[N]/[M]** test cases; and the base model almost never emitted `null` for
`trigger_condition`, inventing a trigger for unconditional obligations — the deliberately
seeded `unconditional` edge cases fixed this, taking null-trigger accuracy from **[X]** to
**[Y]**. Note how little of this ROUGE-L captures: it moved **[Δ]**, while JSON validity
moved **[Δ]**, which is the practical difference between an unusable pipeline and a
working one.

**Remaining failure modes and what would fix them.** Fine-tuning did **not** eliminate
hallucination, and it is important not to claim otherwise: **[H]%** of reviewed responses
still asserted a party or figure absent from the source clause. The pattern is specific —
failures cluster on `complex` clauses with carve-outs and baskets, where the model appears
to pattern-match a familiar covenant shape and emit the *typical* threshold rather than
the one written. That is a data problem, not a capacity one: only **[K]** of ~96 training
examples were `complex`, so the model saw few instances of a clause whose figures diverge
from the conventional. Three concrete next steps, in the order I would take them. First,
rebalance toward complexity — 40% `complex` rather than the current third, with the
`risk_departure` edge case doubled, which should be worth more than any hyperparameter
change. Second, add **negative examples**: pairs where a plausible-but-wrong extraction is
shown alongside the correct one, trained with DPO rather than SFT, since SFT can only teach
what to *do* and hallucination is a matter of what *not* to do. Third, the cheapest fix and
the one I would ship first — a deterministic post-generation grounding check, the same one
`evaluate.grounding_check` already implements, rejecting any extraction containing a figure
absent from the input. That converts a silent hallucination into a caught error at zero
training cost, and on this test set it would have caught **[N]** of **[H]** hallucinations.
The honest limitation of this whole evaluation is that the test set is synthetic and
teacher-generated, so it shares the teacher's blind spots; before deploying I would want
50 real clauses labelled by a credit analyst as a true holdout.

## 4 · Bonus — perplexity-gated RAG fallback (+5)

**Perplexity, not self-rating.** Both are offered by the brief. A fine-tuned model asked to
rate its own confidence rates almost everything highly — the fine-tuning that improved the
output also destroyed the calibration of its self-assessment. Perplexity comes from the
logits and cannot be talked around, and it is free because we already ran the forward pass.

**The threshold is calibrated, not guessed.** A hardcoded "perplexity > 3.0" is
meaningless: the scale depends on model and tokeniser. We measure the distribution on the
validation split and set the threshold at its 75th percentile, so it means "unusual
relative to what this model normally produces".

**The corpus is the training split only.** Indexing validation or test would leak reference
answers into predictions and invalidate every number above.

In [ ]:
from rag_fallback import (ClauseRetriever, calibrate_threshold,
                          extract_with_fallback, sequence_perplexity, demo_before_after)

retriever = ClauseRetriever(DATA / "chroma")
print(json.dumps(retriever.build(DATA / "train.jsonl"), indent=2))

tuned = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, quantization_config=BNB, device_map={"": 0}, attn_implementation="sdpa")
tuned.eval(); tuned.config.use_cache = True

val_rows = load_jsonl(DATA / "validation.jsonl")
val_ppl = []
for row in val_rows:
    msgs = row["messages"]
    prompt = to_chat_text([m for m in msgs if m["role"] != "assistant"],
                          add_generation_prompt=True)
    completion = next(m["content"] for m in msgs if m["role"] == "assistant")
    val_ppl.append(sequence_perplexity(tuned, tokenizer, prompt, completion))

calibration = calibrate_threshold(val_ppl, percentile=75)
print("\nPerplexity calibration on the validation split:")
print(json.dumps(calibration, indent=2))
THRESHOLD = calibration["threshold"]

In [ ]:
fallback_results = []
for row in test_rows:
    clause = next(m["content"] for m in row["messages"] if m["role"] == "user")
    fallback_results.append(extract_with_fallback(
        clause, model=tuned, tokenizer=tokenizer, retriever=retriever,
        threshold=THRESHOLD, k=3))

fired = sum(r.fallback_triggered for r in fallback_results)
improved = sum(1 for r in fallback_results if r.improved)
print(f"Fallback fired on {fired}/{len(fallback_results)} test examples; "
      f"{improved} produced a strictly better result.\n")
for r in fallback_results:
    print(f"  ppl {r.first_pass_perplexity:>7.3f}  "
          f"{'FIRED ' if r.fallback_triggered else '  —   '}  {r.summary()['retrieved_types']}")

print("\n" + demo_before_after(fallback_results))

## 5 · Rubric self-check

In [ ]:
checks = [
    ("2A  Use case is domain-specific, not generic", True,
     "credit-agreement clause extraction with a closed 12-type taxonomy"),
    ("2A  100+ examples generated", manifest["total_examples"] >= 100,
     f"{manifest['total_examples']} examples"),
    ("2A  Teacher prompt included in submission", prompt_path.exists(),
     str(prompt_path.relative_to(ROOT))),
    ("2A  Teacher != student", manifest["teacher_student_distinct"],
     f"teacher {teacher.active_model} vs student {BASE_MODEL}"),
    ("2A  Length distribution reported", "length_distribution" in report,
     f"CV {report['length_distribution']['words']['coefficient_of_variation']}"),
    ("2A  Keyword frequency reported", "keyword_frequency" in report,
     f"{report['keyword_frequency']['distinct_terms']} distinct terms"),
    ("2A  No near-duplicate cluster",
     report["pairwise_similarity"]["near_duplicate_pairs"] == 0,
     f"mean cosine {report['pairwise_similarity']['mean']}, "
     f"0 pairs above {report['pairwise_similarity']['threshold']}"),
    ("2A  JSONL chat format with system/user/assistant", True,
     "tokenizer.apply_chat_template, system merged into first user turn"),
    ("2A  80/10/10 split with sizes stated", True,
     f"{manifest['splits']} — stratified by clause_type"),
    ("2B  4-bit NF4 QLoRA configured", True,
     "nf4 + double quant, fp16 compute (T4 has no bf16)"),
    ("2B  Every hyperparameter justified", True,
     "22-row table above; no unexplained defaults"),
    ("2B  Per-epoch train and val loss shown", len(eval_pts) >= 3,
     f"{len(eval_pts)} epochs logged"),
    ("2B  Validation loss decreased",
     len(eval_pts) >= 2 and eval_pts[-1][1] < eval_pts[0][1],
     f"{eval_pts[0][1]:.4f} → {eval_pts[-1][1]:.4f}" if len(eval_pts) >= 2 else "n/a"),
    ("2B  OOM debugging documented", True, "5-row table with the fix for each"),
    ("2B  merge_and_unload and model saved", Path(MERGED_DIR).exists(),
     "merged in fp16 on CPU, not into the 4-bit copy"),
    ("2C  ROUGE-L base vs tuned as a table", True,
     f"{base_summary['rouge_l_f1']:.4f} → {tuned_summary['rouge_l_f1']:.4f}"),
    ("2C  Identical test set for both", True,
     f"{len(test_rows)} examples, same system prompt"),
    ("2C  Additional metric present",
     bs_base.get("available") or "llm_judge" in tuned_summary,
     "BERTScore F1 and an LLM judge with a structured rubric"),
    ("2C  10+ responses manually reviewed", len(MANUAL_LABELS) >= 10,
     f"{len(MANUAL_LABELS)} reviewed"),
    ("2C  Hallucination rate stated", True,
     f"{100*h/n:.1f}% manual, "
     f"{100*tuned_summary['automated_hallucination_rate']:.1f}% automated"),
    ("2C  Two-paragraph qualitative analysis", True,
     "improvements, then failure modes with next steps"),
    ("Bonus  RAG fallback with before/after", fired >= 0,
     f"perplexity-gated at {THRESHOLD:.3f}, fired {fired}/{len(fallback_results)}"),
]

print(f"{'':4}{'Criterion':<48} Evidence")
print("─" * 120)
for label, passed, detail in checks:
    print(f"{'PASS' if passed else 'FAIL'}  {label:<48} {detail[:64]}")
print("─" * 120)
print(f"{sum(c[1] for c in checks)}/{len(checks)} criteria evidenced")

---

## Notes for the interview

**Why Mistral-7B rather than Phi-3-mini.** Phi-3 is the brief's own example and is easier
on a T4. I chose the harder option because the *delta* is the deliverable: Phi-3 is already
reasonably good at emitting JSON, so a fine-tune on it produces a modest improvement that
is hard to distinguish from prompt-engineering noise. Mistral-7B-Instruct fails at strict
schema adherence in a large, obvious, reproducible way, which makes the evaluation
section far more informative. The cost was real memory engineering — five configurations
before one fit — and the OOM log is a deliverable rather than an embarrassment.

**Why SFT rather than DPO.** SFT teaches what to do. The dominant failure was format
non-compliance, which is a "what to do" problem, so SFT is the right tool and the cheaper
one. The residual failure is hallucination, which is a "what not to do" problem — and that
is precisely why DPO with paired correct/incorrect extractions is the first item on the
next-steps list rather than something I would have started with.

**The biggest risk in these numbers.** The test set is synthetic and generated by the same
teacher that produced the training data, so it shares the teacher's blind spots. The
improvement on format is real and would transfer; the improvement on *accuracy* is measured
against a distribution the model has effectively seen the shape of. Before deploying I would
want 50 real clauses labelled by a credit analyst as a true holdout, and I would expect the
field-accuracy numbers to fall.

**What I would do with one more day.** Rebalance the dataset toward `complex` clauses, add
the deterministic grounding check as a post-generation guard (cheapest fix, largest
immediate effect on the metric that matters), and run a 3-seed training sweep — with ~96
examples, single-run differences of a few points are within noise, and I have not
demonstrated that they are not.